# 📈 ViHateT5 Improvement Results

A comprehensive summary of model improvements for Vietnamese Hate Speech Detection.

**Contents:**
1. Focal Loss vs Cross-Entropy
2. Model Ensemble (Weighted + Majority Voting)
3. Error Analysis & McNemar Significance Tests
4. Combined Comparison
5. Conclusions

> **Note:** This notebook uses pre-computed results only. No GPU or model inference required.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from IPython.display import Image, display

RESULTS_DIR = Path("../results")
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Focal Loss vs Cross-Entropy

Comparison between standard Cross-Entropy loss and Focal Loss (γ=2, α=0.25) on the ViHSD dataset.
Focal Loss helps the model focus on hard-to-classify samples, which is especially effective for imbalanced data.

In [ ]:
df_focal = pd.read_csv(RESULTS_DIR / "focal_loss_comparison.csv")
display(df_focal.style.format({
    "f1_macro": "{:.4f}", "accuracy": "{:.4f}",
    "f1_clean": "{:.4f}", "f1_offensive": "{:.4f}", "f1_hate": "{:.4f}"
}).set_caption("Focal Loss vs Cross-Entropy on ViHSD"))

# Calculate improvement
ce_f1 = df_focal.loc[df_focal["loss_type"] == "CrossEntropy", "f1_macro"].values[0]
fl_f1 = df_focal.loc[df_focal["loss_type"] == "FocalLoss", "f1_macro"].values[0]
print(f"\n📊 Macro F1 Improvement: {ce_f1:.4f} → {fl_f1:.4f} (+{(fl_f1-ce_f1)/ce_f1*100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ["accuracy", "f1_macro", "f1_clean", "f1_hate"]
x = np.arange(len(metrics))
width = 0.35

ce_vals = [df_focal.loc[df_focal["loss_type"]=="CrossEntropy", m].values[0] for m in metrics]
fl_vals = [df_focal.loc[df_focal["loss_type"]=="FocalLoss", m].values[0] for m in metrics]

ax.bar(x - width/2, ce_vals, width, label="Cross-Entropy", color="#ff6b6b")
ax.bar(x + width/2, fl_vals, width, label="Focal Loss", color="#4ecdc4")
ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Macro F1", "F1 Clean", "F1 Hate"])
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Cross-Entropy vs Focal Loss — ViHSD")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Model Ensemble Results

Combining predictions from multiple models (T5 + BERT) using Weighted Voting and Majority Voting strategies.

In [ ]:
df_ensemble = pd.read_csv(RESULTS_DIR / "ensemble_results.csv")
display(df_ensemble.style.format({
    "Accuracy": "{:.4f}", "Macro_F1": "{:.4f}",
    "F1_CLEAN": "{:.4f}", "F1_OFFENSIVE": "{:.4f}", "F1_HATE": "{:.4f}"
}).set_caption("Ensemble Results on ViHSD"))

## 3. McNemar Statistical Significance Tests

McNemar's test comparing each model pair — determining whether performance differences are statistically significant (p < 0.05).

In [ ]:
df_mcnemar = pd.read_csv(RESULTS_DIR / "analysis" / "mcnemar_results_vihsd.csv")
df_mcnemar["p_value"] = df_mcnemar["p_value"].apply(lambda x: f"{x:.6f}")
display(df_mcnemar.style.set_caption("McNemar Test Results — ViHSD"))

## 4. Combined Model Comparison

A grouped bar chart comparing all models across key metrics.

In [ ]:
img_path = RESULTS_DIR / "images" / "combined_comparison.png"
if img_path.exists():
    display(Image(filename=str(img_path), width=800))
else:
    print(f"⚠️ Image not found: {img_path}")

## 5. Conclusions

| Method | Macro F1 | Assessment |
|--------|----------|------------|
| Baseline (CE) | 0.5198 | Weak on minority classes |
| Focal Loss | 0.7478 | **Best improvement** (+43.9%) |
| Ensemble (Weighted) | 0.6346 | Better than baseline |
| Ensemble (Majority) | 0.5317 | Comparable to baseline |

**Key findings:**
- Focal Loss yields the largest Macro F1 improvement by focusing on hard examples
- Ensemble voting improves over baseline but does not surpass single-model Focal Loss
- McNemar tests confirm statistically significant differences between key model pairs